# Лабораторная работа 5. Позиционное кодирование и RoPE (версия преподавателя с решениями)

**Курс:** Методы обработки естественного языка · **Блок 2:** attention и архитектура трансформера

**Цель.** Понять, как трансформер получает информацию о порядке токенов и почему модели плохо работают на длинах контекста, которых не видели при обучении.

**Что нужно сдать:**
1. Единый интерфейс для пяти вариантов: NoPE, синусоидального, обучаемого, ALiBi и RoPE.
2. Численную проверку относительности RoPE и график затухания верхней границы.
3. Таблицу «кодирование × длина контекста» для символьной LM (экстраполяция длины).
4. Position interpolation и NTK-aware scaling для RoPE без дообучения и после короткого дообучения.
5. Тест «иголка в стоге сена» на синтетической задаче поиска.

**Теория.** RoPE поворачивает пары координат \((x_{2i}, x_{2i+1})\) на угол \(m\theta_i\), где \(\theta_i = b^{-2i/d}\), а \(m\) — позиция. В комплексной записи \(\tilde q_m = q\,e^{im\theta}\), \(\tilde k_n = k\,e^{in\theta}\), поэтому
\[
\mathrm{Re}\langle \tilde q_m, \tilde k_n\rangle = \mathrm{Re}\sum_i q_i \bar k_i\, e^{i(m-n)\theta_i},
\]
то есть скалярное произведение зависит только от разности позиций \(m - n\).

**Расширение контекста в \(s\) раз:**
- position interpolation: \(m \mapsto m/s\);
- NTK-aware scaling: \(b \mapsto b\, s^{d/(d-2)}\) — высокие частоты почти не меняются, низкие интерполируются.

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

## 1. Позиционные кодирования

In [ ]:
def sinusoidal_pe(T, d):
    pos = torch.arange(T).float()[:, None]; i = torch.arange(0, d, 2).float()
    ang = pos / (10000 ** (i / d))
    pe = torch.zeros(T, d); pe[:, 0::2] = torch.sin(ang); pe[:, 1::2] = torch.cos(ang)
    return pe

def rope_angles(T, d, base=10000.0, scale=1.0):
    """Углы m*theta_i, форма (T, d/2). scale > 1 — position interpolation."""
    inv = base ** (-torch.arange(0, d, 2).float() / d)
    return torch.outer(torch.arange(T).float() / scale, inv)

def apply_rope(x, ang):
    """x: (..., T, d), ang: (T, d/2). Поворачивает пары (x[2i], x[2i+1])."""
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)

def alibi_bias(T, n_head):
    """Смещение -m_h * (i - j) для j <= i; m_h = 2^{-8h/H}, h = 1..H. Форма (H, T, T)."""
    slopes = torch.tensor([2 ** (-8 * (h + 1) / n_head) for h in range(n_head)])
    dist = torch.arange(T)[:, None] - torch.arange(T)[None, :]
    return -slopes[:, None, None] * dist.clamp(min=0).float()

### Проверка относительности RoPE

Для случайных \(q, k\) проверьте, что \(\langle R_m q, R_n k\rangle\) зависит только от \(m - n\), и постройте \(\frac{1}{d/2}\big|\sum_i e^{i r \theta_i}\big|\) в зависимости от расстояния \(r\) — величину, определяющую затухание верхней границы скалярного произведения при одинаковых \(q, k\).

In [ ]:
d = 64; torch.manual_seed(0); q, k = torch.randn(d), torch.randn(d)
ang = rope_angles(2048, d)
def rot_dot(m, n): return (apply_rope(q[None], ang[m:m + 1])[0] @ apply_rope(k[None], ang[n:n + 1])[0]).item()
for (m, n) in [(5, 2), (105, 102), (1005, 1002)]:
    print(f"m={m:4d}, n={n:4d}: <R_m q, R_n k> = {rot_dot(m, n):.6f}")
assert abs(rot_dot(5, 2) - rot_dot(1005, 1002)) < 1e-3

plt.figure(figsize=(7, 3.5))
for base in (10000, 500):
    th = base ** (-torch.arange(0, d, 2).float() / d); r = torch.arange(2048).float()
    val = torch.complex(torch.cos(torch.outer(r, th)), torch.sin(torch.outer(r, th))).mean(1).abs()
    plt.plot(r, val, label=f"base={base}", lw=0.8)
plt.xscale("log"); plt.xlabel("относительное расстояние r"); plt.ylabel("|mean exp(i r θ)|"); plt.legend()
plt.title("Затухание RoPE с расстоянием"); plt.show()

## 2. Трансформер с переключаемым позиционным кодированием

In [ ]:
class PEAttention(nn.Module):
    def __init__(self, d_model, n_head, pe):
        super().__init__()
        self.h, self.dh, self.pe = n_head, d_model // n_head, pe
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False); self.o = nn.Linear(d_model, d_model, bias=False)
        self.rope_base, self.rope_scale = 10000.0, 1.0
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = (t.view(B, T, self.h, self.dh).transpose(1, 2) for t in self.qkv(x).split(C, 2))
        if self.pe == "rope":
            ang = rope_angles(T, self.dh, self.rope_base, self.rope_scale).to(x.device)
            q, k = apply_rope(q, ang), apply_rope(k, ang)
        if self.pe == "alibi":
            bias = alibi_bias(T, self.h).to(x.device)
            causal = torch.ones(T, T, dtype=torch.bool, device=x.device).triu(1)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.masked_fill(causal, float("-inf")))
        else:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).reshape(B, T, C))

class PETransformer(nn.Module):
    def __init__(self, vocab, pe, d_model=128, n_head=4, n_layer=2, max_T=64, mlp=True):
        super().__init__()
        self.pe, self.max_T = pe, max_T
        self.emb = nn.Embedding(vocab, d_model)
        if pe == "learned": self.pos = nn.Parameter(torch.randn(max_T, d_model) * 0.02)
        self.attn = nn.ModuleList([PEAttention(d_model, n_head, pe) for _ in range(n_layer)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layer)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layer)])
        self.mlp = nn.ModuleList([nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
                                  if mlp else nn.Identity() for _ in range(n_layer)])
        self.use_mlp = mlp
        self.lnf, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, vocab)
    def forward(self, x):
        T = x.size(1); h = self.emb(x)
        if self.pe == "sinusoidal": h = h + sinusoidal_pe(T, h.size(-1)).to(x.device)
        if self.pe == "learned":
            if T > self.max_T: raise ValueError("обучаемые позиции не определены за пределами обучающей длины")
            h = h + self.pos[:T]
        for a, l1, l2, m in zip(self.attn, self.ln1, self.ln2, self.mlp):
            h = h + a(l1(h))
            if self.use_mlp: h = h + m(l2(h))
        return self.head(self.lnf(h))
    def set_rope(self, base=10000.0, scale=1.0):
        for a in self.attn: a.rope_base, a.rope_scale = base, scale

PES = ["nope", "sinusoidal", "learned", "alibi", "rope"]

## 3. Экстраполяция длины на символьной LM

Обучите модели на окнах длины `T_TRAIN` и измерьте среднюю кросс-энтропию на окнах длины \(T_{train}\), \(2T_{train}\), \(4T_{train}\), \(8T_{train}\). Отдельно посчитайте loss только на позициях \(\ge T_{train}\) — это и есть качество экстраполяции.

In [ ]:
text = load_shakespeare(); chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text]); ntr = int(0.9 * len(data)); train_d, val_d = data[:ntr], data[ntr:]
T_TRAIN = 64 if FAST_DEV_RUN else 128
STEPS = 400 if FAST_DEV_RUN else 4000

def lm_batch(d, T, B=32):
    ix = torch.randint(len(d) - T - 1, (B,))
    return torch.stack([d[i:i + T] for i in ix]).to(DEVICE), torch.stack([d[i + 1:i + T + 1] for i in ix]).to(DEVICE)

def train_lm(model, steps, T, lr=2e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    for s in range(steps):
        x, y = lm_batch(train_d, T); loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    return model

@torch.no_grad()
def eval_lengths(model, lengths, n_windows=16):
    model.eval(); out = {}
    g = torch.Generator().manual_seed(0)
    for L in lengths:
        ix = torch.randint(len(val_d) - L - 1, (n_windows,), generator=g)
        x = torch.stack([val_d[i:i + L] for i in ix]).to(DEVICE); y = torch.stack([val_d[i + 1:i + L + 1] for i in ix]).to(DEVICE)
        try:
            ce = F.cross_entropy(model(x).flatten(0, 1), y.flatten(), reduction="none").view(n_windows, L)
            out[L] = (ce.mean().item(), ce[:, T_TRAIN:].mean().item() if L > T_TRAIN else float("nan"))
        except ValueError:
            out[L] = (float("nan"), float("nan"))
    model.train(); return out

lengths = [T_TRAIN * m for m in (1, 2, 4, 8)]
lm_models, rows = {}, []
for pe in PES:
    set_seed(); t0 = time.time()
    lm_models[pe] = train_lm(PETransformer(len(chars), pe, max_T=T_TRAIN).to(DEVICE), STEPS, T_TRAIN)
    res = eval_lengths(lm_models[pe], lengths)
    rows.append({"PE": pe, **{f"L={L}": round(v[0], 3) for L, v in res.items()}, **{f"хвост L={L}": round(v[1], 3) for L, v in res.items() if L > T_TRAIN}})
    print(pe, f"{time.time() - t0:.0f} c")
import pandas as pd
pd.DataFrame(rows).set_index("PE")

## 4. Расширение контекста RoPE: position interpolation и NTK-aware scaling

Для RoPE-модели при длине \(L = s\,T_{train}\) сравните:
1. исходную модель;
2. position interpolation (`scale = s`);
3. NTK-aware scaling (`base = 10000 · s^{d/(d-2)}`).

Затем дообучите копии модели на длине \(L\) в течение короткого числа шагов (около 5% исходного обучения) с каждым вариантом и сравните снова.

In [ ]:
import copy
s_ext = 4; L_ext = s_ext * T_TRAIN; d_head = 128 // 4
variants = {"исходная": dict(base=10000.0, scale=1.0), "PI": dict(base=10000.0, scale=float(s_ext)),
            "NTK": dict(base=10000.0 * s_ext ** (d_head / (d_head - 2)), scale=1.0)}
rows = []
for name, kw in variants.items():
    m = copy.deepcopy(lm_models["rope"]); m.set_rope(**kw)
    before = eval_lengths(m, [L_ext])[L_ext]
    train_lm(m, max(20, STEPS // 20), L_ext, lr=5e-4)
    after = eval_lengths(m, [L_ext])[L_ext]
    rows.append({"вариант": name, "без дообучения: всё окно": before[0], "без дообучения: хвост": before[1],
                 "после дообучения: всё окно": after[0], "после дообучения: хвост": after[1]})
display(pd.DataFrame(rows).set_index("вариант").round(3))

## 5. «Иголка в стоге сена»

Задача: последовательность пар `ключ значение` и в конце запрос-ключ; нужно вернуть значение (как в ЛР 4). Обучите attention-only модели (2 слоя, без MLP) на длине 32 и проверьте точность на длинах 32, 64, 128 (в полном режиме и 256). Запрос может относиться к паре в любом месте контекста.

In [ ]:
VOCAB_KV = 64

def needle_batch(T, B=128):
    n = (T - 1) // 2
    keys = torch.stack([torch.randperm(VOCAB_KV // 2)[:n] if n <= VOCAB_KV // 2 else torch.randint(VOCAB_KV // 2, (n,)) for _ in range(B)])
    vals = torch.randint(VOCAB_KV // 2, VOCAB_KV, (B, n))
    j = torch.randint(n, (B,))
    x = torch.cat([torch.stack([keys, vals], 2).flatten(1), keys[torch.arange(B), j][:, None]], 1)
    # при повторяющихся ключах (длинные контексты) ответ — значение последнего вхождения ключа
    tgt = vals[torch.arange(B), j].clone()
    for b in range(B):
        last = (keys[b] == keys[b, j[b]]).nonzero().max()
        tgt[b] = vals[b, last]
    return x.to(DEVICE), tgt.to(DEVICE)

T_NEEDLE = 33
test_T = [33, 65, 129] if FAST_DEV_RUN else [33, 65, 129, 257]
rows = []
for pe in PES:
    set_seed(); m = PETransformer(VOCAB_KV, pe, n_layer=2, max_T=T_NEEDLE, mlp=False).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for s in range(800 if FAST_DEV_RUN else 4000):
        x, y = needle_batch(T_NEEDLE); loss = F.cross_entropy(m(x)[:, -1], y)
        opt.zero_grad(); loss.backward(); opt.step()
    r = {"PE": pe}
    with torch.no_grad():
        for T in test_T:
            try:
                x, y = needle_batch(T, B=512); r[f"T={T}"] = (m(x)[:, -1].argmax(-1) == y).float().mean().item()
            except ValueError:
                r[f"T={T}"] = float("nan")
    rows.append(r); print(r)
pd.DataFrame(rows).set_index("PE").round(3)

## Вопросы

1. Докажите свойство относительности RoPE. Почему RoPE применяется к \(q\) и \(k\), но не к \(v\)?
2. Почему обучаемые позиционные эмбеддинги в принципе не экстраполируют, а синусоидальные формально определены, но на практике тоже плохо экстраполируют?
3. Как NoPE получает позиционную информацию в каузальной модели?
4. Объясните разницу между PI и NTK-aware scaling в терминах частот \(\theta_i\). Какие частоты «ломаются» при экстраполяции?
5. Как ALiBi реализует индуктивное смещение к локальности и почему оно мешает задаче поиска далёкой «иголки»?

> **Для преподавателя.**
>
> **Ожидаемые результаты (FAST_DEV_RUN, около 4 минут на CPU; T_TRAIN = 64).** На длине обучения кросс-энтропия около 2.0–2.4 (у NoPE и learned хуже: короткое обучение). При удлинении до 8×: learned не определено, у sinusoidal хвост растёт до 3.2–3.4, у RoPE — с 2.2 до 3.0, ALiBi почти не деградирует (около 2.0–2.1), NoPE деградирует слабо, но и стартует с худшего уровня. Расширение RoPE в 4 раза без дообучения: NTK улучшает хвост (около 2.17 против 2.53 у исходной), PI без дообучения хуже исходной (около 3.1): сжатие высоких частот ломает локальные паттерны. После 20 шагов дообучения на длине 256: NTK около 1.85, исходная около 1.89, PI около 2.13 — PI требует более длительного дообучения (в статье о PI — порядка 1000 шагов). Поиск «иголки» за 800 шагов: RoPE и ALiBi решают задачу на обучающей длине (0.93–0.99); NoPE, sinusoidal и learned не успевают выучить механизм (0.15–0.35). На длине 129 точность падает у всех; ALiBi держится лучше (около 0.55), RoPE — около 0.26. Конкретные числа зависят от seed, поэтому в полном режиме требуется усреднение по 3 seed.
>
> **Ответы.**
> 1. См. введение: \(\langle R_m q, R_n k\rangle = q^\top R_m^\top R_n k = q^\top R_{n-m} k\), так как повороты образуют группу. \(V\) не участвует в вычислении весов внимания; поворот \(v\) смешал бы содержимое с позицией без пользы для относительности.
> 2. Обучаемая таблица не содержит строк для новых позиций. Синусоиды определены, но модель не видела сочетаний фаз низкочастотных компонент на больших позициях — это сдвиг распределения входа.
> 3. Каузальная маска делает число доступных токенов зависящим от позиции; например, равномерное внимание к началу последовательности даёт сигнал, убывающий как \(1/t\), из которого можно восстановить позицию.
> 4. PI сжимает все частоты одинаково, в том числе высокие, нужные для различения соседних токенов. NTK-aware почти не меняет высокие частоты и интерполирует низкие, у которых период больше обучающего контекста. Ломаются именно низкие частоты: полный оборот они делают только на длинах больше обучающей.
> 5. Линейный штраф по расстоянию в логитах экспоненциально подавляет веса далёких токенов, поэтому модель смещена к локальному вниманию.